# CatfishCare: Smart Aquaculture AI Training
Jupyter Notebook ini dirancang untuk melatih model **Bidirectional LSTM** untuk memprediksi kualitas air kolam lele (Suhu, Turbiditas, PH) 24 jam ke depan berdasarkan data 48 jam sebelumnya.

## 1. Install Required Dependencies
Langkah ini untuk mengatasi `ModuleNotFoundError`. Jalankan sel di bawah ini untuk memastikan semua library tersedia.

In [1]:
!pip install pandas numpy scikit-learn tensorflow joblib

  Using cached tensorflow-2.21.0-cp310-cp310-win_amd64.whl.metadata (4.5 kB)
  Using cached keras-3.12.2-py3-none-any.whl.metadata (5.9 kB)
Using cached tensorflow-2.21.0-cp310-cp310-win_amd64.whl (350.6 MB)
Using cached keras-3.12.2-py3-none-any.whl (1.5 MB)

   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   ---------------------------------------- 0/2 [keras]
   --------

## 2. Import Libraries and Global Configuration
Konfigurasi parameter seperti `LOOKBACK` (48 jam) dan `HORIZON` (24 jam).

In [2]:
import pandas as pd
import numpy as np
import os
import glob
import joblib
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, RepeatVector, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping

# Konfigurasi
DATASET_PATH = 'datasets/'
SCALER_FILENAME = 'scaler_catfishcare.pkl'
MODEL_FILENAME = 'model_catfishcare_bilstm.keras'
LOOKBACK = 48
HORIZON = 24
FEATURES = ['Temperature (C)', 'Turbidity(NTU)', 'PH']

## 3. Data Loading and Merging
Membaca semua file `IoTPond*.csv` dari folder `datasets/`.

In [3]:
def load_and_merge_data(path):
    all_files = glob.glob(os.path.join(path, "IoTPond*.csv"))
    df_list = []
    for filename in all_files:
        df = pd.read_csv(filename)
        df_list.append(df)
    return pd.concat(df_list, axis=0, ignore_index=True)

raw_data = load_and_merge_data(DATASET_PATH)
print(f"Data mentah terload: {len(raw_data)} baris.")

Data mentah terload: 1111456 baris.


## 4. Data Preprocessing and Cleaning
Pembersihan zona waktu, konversi datetime, resampling ke per 1 jam, dan interpolasi data kosong.

In [4]:
def preprocess_data(df):
    df['created_at'] = df['created_at'].str.replace(' CET', '', regex=False)
    df['created_at'] = df['created_at'].str.replace(' UTC', '', regex=False)
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    df = df.dropna(subset=['created_at'])
    df.set_index('created_at', inplace=True)
    df.sort_index(inplace=True)
    df = df[FEATURES]
    df_resampled = df.resample('1H').mean()
    df_resampled = df_resampled.interpolate(method='linear').dropna()
    return df_resampled

clean_data = preprocess_data(raw_data)
print(f"Data setelah cleaning: {len(clean_data)} jam.")

Data setelah cleaning: 3233 jam.


C:\Users\USER\AppData\Local\Temp\ipykernel_19232\3712357351.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_index(inplace=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_19232\3712357351.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_resampled = df.resample('1H').mean()


## 5. Feature Normalization
Menggunakan `MinMaxScaler` untuk menormalisasi Suhu, Turbidity, dan PH ke skala 0-1.

In [5]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(clean_data)
# Simpan scaler untuk nantinya
joblib.dump(scaler, SCALER_FILENAME)
print("Scaler telah disimpan.")

Scaler telah disimpan.


## 6. Creating Time-Series Sequences
Mentransformasi data menjadi format 3D (samples, lookback, features).

In [6]:
def create_sequences(data, lookback, horizon):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : i + lookback])
        y.append(data[i + lookback : i + lookback + horizon])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, LOOKBACK, HORIZON)
print(f"Shape X: {X.shape}, Shape y: {y.shape}")

Shape X: (3162, 48, 3), Shape y: (3162, 24, 3)


## 7. Chronological Data Splitting
Memecah data menjadi 80% Training dan 20% Testing tanpa mengacak urutan waktu.

In [7]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Train samples: 2529, Test samples: 633


## 8. Build Bidirectional LSTM Architecture
Membuat model dengan 2 layer BiLSTM dan output multi-step forecasting.

In [8]:
model = Sequential([
    Bidirectional(LSTM(64, activation='relu', return_sequences=True), 
                  input_shape=(LOOKBACK, len(FEATURES))),
    Bidirectional(LSTM(32, activation='relu', return_sequences=False)),
    RepeatVector(HORIZON),
    LSTM(32, activation='relu', return_sequences=True),
    TimeDistributed(Dense(len(FEATURES)))
])

model.compile(optimizer='adam', loss='mae')
model.summary()

c:\Users\USER\anaconda3\envs\catfish_env\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 48, 128)        │        34,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 24, 32)         │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 24, 3)          │            99 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,547 (345.89 KB)

 Trainable params: 88,547 (345.89 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Model Compilation and Training
Training selama 50 epoch dengan EarlyStopping.

In [9]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 48ms/step - loss: 0.2023 - val_loss: 0.1730
Epoch 2/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.0767 - val_loss: 0.4508
Epoch 3/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - loss: 0.0649 - val_loss: 0.3646
Epoch 4/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - loss: 0.0562 - val_loss: 0.1344
Epoch 5/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 0.0553 - val_loss: 0.1293
Epoch 6/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 0.0540 - val_loss: 0.1678
Epoch 7/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - loss: 0.0512 - val_loss: 0.1211
Epoch 8/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 0.0510 - val_loss: 0.1250
Epoch 9/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 0.0496 - val_loss: 0.1162
Epoch 10/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.0501 - val_loss: 0.1179
Epoch 11/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.0491 - val_loss: 0.2961
Epoch 12/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.

## 10. Model and Scaler Persistence
Menyimpan model akhir yang sudah terlatih.

In [10]:
model.save(MODEL_FILENAME)
print(f"Model berhasil disimpan sebagai {MODEL_FILENAME}")

Model berhasil disimpan sebagai model_catfishcare_bilstm.keras
